# Melt Budget Data Loading
**Notebook 1 of 2** — run before `sankey_figures.ipynb`.

Loads, area-integrates, and saves CMIP6 and CESM2-LE ice and snow mass budget variables as `.nc` files to `save_path`. Processes one CMIP6 model at a time (set `models` below). The saved files are read directly by `sankey_figures.ipynb`.

In [1]:
# Install these packages once per session on the cryohub.
# All packages are used in the CMIP6 and CESM loading code: files/load.py
# Skip if these packages are already installed in your environment

# ── Core climate/geospatial dependencies ──────────────────────────────────────
%pip install -q regionmask       # regional masking for geospatial data
%pip install -q bottleneck       # speeds up xarray reduce operations (rolling mean, etc.)
%pip install -q xesmf            # regridding for Earth System Model output
%pip install -q cf_xarray        # CF convention accessor for xarray datasets

# ── ESGF data access ──────────────────────────────────────────────────────────
%pip install -q esgf-pyclient    # search client for the Earth System Grid Federation
%pip install -q intake-esgf      # intake driver for ESGF catalogs

# ── Version-pinned installs (workarounds) ─────────────────────────────────────
# importlib_metadata<8 fixes an import error with esmpy
# may no longer be needed    
#%pip install -q "importlib_metadata<8"

# globus-sdk must stay below v4 until intake-esgf adds compatibility
%pip install -q "globus-sdk<4"

# pydantic upgrade needed for intake-esm compatibility
%pip install -q --upgrade intake-esm pydantic

# numpy pinned to <=2.3 due to numba incompatibility
%pip install -q "numpy<=2.3"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


**Restart the kernel after the installs above before running any further cells.**

In [2]:
import os
# HDF5's file-locking protocol is unreliable over NFS (this repo's save_path/melt_path
# live on an NFS mount) and intermittently raises 'RuntimeError: NetCDF: HDF error' on
# to_netcdf()/open_dataset() calls — disable it before any netCDF4/xarray I/O.
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import sys
import logging
from io import StringIO
import numpy as np
import pandas as pd
import xarray as xr
import intake
from dask.diagnostics import ProgressBar
from load import CMIP6, CESM
from functions import region_mask,preferred_load_list,to_pystr_list,NH_seaice_regions
import time
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from functools import reduce

In [3]:
# Directory where area-integrated .nc files are written (must match melt_path in sankey_figures.ipynb)
save_path = '/home/jovyan/shared-public/ICESat-2-sea-ice/better_output/melt/'
save = True  # set True to write files
process_CESM2_LE = True  # set True to also (re)process CESM2-LE this run — separate from `models`
                          # above; loads from local storage/THREDDS OPeNDAP, not ESGF/Pangeo, and is
                          # heavy (50 members), so leave off unless you actually need it regenerated.

cat_url  = 'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'
cat_url2 = 'https://storage.googleapis.com/cmip6/cmip6-pgf-ingestion-test/catalog/catalog.json'

args = {
    'verbose': True,
    'skip_sids': ['CAS-ESM2-0', 'GISS-E2-1-G-CC', 'GISS-E2-2-G', 'GISS-E2-1-G', 'GISS-E2-1-H'],  # models with known data issues
    'grid_label': ['gn', 'gr'],
    'members': 'all',
    'experiment_id': ['ssp245'],
    'time_chunks': 200,       # months per dask chunk
    'esgf_url': 'https://esgf-node.ornl.gov/esg-search',
    'end_year': 2034,        # skip ESGF file chunks that start after this year — the chunk covering our
                             # end date still gets downloaded in full (chunks can't be partially fetched), so
                             # this doesn't guarantee no data past 2034, just skips chunks with none at all
    'cat_url': cat_url,
    'cat_url2': cat_url2,
}# Directory where area-integrated .nc files are written (must match melt_path in sankey_figures.ipynb)
save_path = '/home/jovyan/shared-public/ICESat-2-sea-ice/better_output/melt/'
save = True  # set True to write files

cat_url  = 'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'
cat_url2 = 'https://storage.googleapis.com/cmip6/cmip6-pgf-ingestion-test/catalog/catalog.json'

args = {
    'verbose': True,
    'skip_sids': ['CAS-ESM2-0', 'GISS-E2-1-G-CC', 'GISS-E2-2-G', 'GISS-E2-1-G', 'GISS-E2-1-H'],  # models with known data issues
    'grid_label': ['gn', 'gr'],
    'members': 'all',
    'experiment_id': ['ssp245'],
    'time_chunks': 200,       # months per dask chunk
    'esgf_url': 'https://esgf-node.ornl.gov/esg-search',
    'end_year': 2034,        # skip ESGF file chunks that start after this year — the chunk covering our
                             # end date still gets downloaded in full (chunks can't be partially fetched), so
                             # this doesn't guarantee no data past 2034, just skips chunks with none at all
    'cat_url': cat_url,
    'cat_url2': cat_url2,
}


In [4]:
process_CMIP6 = False  # set False to skip the whole CMIP6 section and only (re)process CESM2-LE below

models = ['EC-Earth3']  # currently only works when set to one model

if isinstance(models, str): models = [models]

In [5]:
variables = ['sidmassmelttop', 'sidmassmeltbot', 'sidmasslat', 'sidmassgrowthbot', 'sidmassgrowthwat', 'sidmasssi', 'sidmassevapsubl', 'sidmassdyn'
             , 'sndmassmelt' , 'sndmasssnf', 'sndmasssi', 'sndmasssubl', 'sndmasswindrif', 'sndmassdyn']

descriptions = [
    '(Sea-Ice Mass Change Through Surface Melting [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Bottom Melting [kg m-2 s-1])',
    '(Lateral Sea Ice Melt Rate [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Basal Growth [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Growth in Supercooled Open Water (Frazil) [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Snow-to-Ice Conversion [kg m-2 s-1])',
    '(Sea-Ice Mass Change Through Evaporation and Sublimation [kg m-2 s-1])',
    '(Sea-Ice Mass Change from Dynamics [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Melt [kg m-2 s-1])',
    '(Snow Mass Change Through Snow Fall [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Snow-to-Ice Conversion [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Evaporation or Sublimation [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Wind Drift of Snow [kg m-2 s-1])',
    '(Snow Mass Rate of Change Through Advection by Sea-Ice Dynamics [kg m-2 s-1])'
]

In [6]:
if process_CMIP6:
    cat_cloud_gn_ssp245 = {}
    cat_esgf_gn_ssp245 = {}

    for var in variables:
        start = time.time()
        cat_cloud_gn_ssp245[var], cat_esgf_gn_ssp245[var] = CMIP6(grid_label=['gn'], members='all', experiment_id = ['ssp245']
                                              , variable=var, table_id='SImon').data_summary()
        elapsed = time.time() - start
        print(f"✅ {var} search completed in {elapsed:.2f} seconds\n")

In [7]:
if process_CMIP6:
    models_gn_ssp245 = {
        var: preferred_load_list(cat_cloud_gn_ssp245[var], cat_esgf_gn_ssp245[var])
        for var in variables
    }

    availability_count = {key: {subkey: len(value) for subkey, value in subdict.items()} for key, subdict in models_gn_ssp245.items()}
    availability_count_df = pd.DataFrame(availability_count).transpose()
    availability_count_df

In [8]:
if process_CMIP6:
    #models with all sea ice mass variables
    reduce(np.intersect1d,[models_gn_ssp245['sidmassmelttop']['esgf'],models_gn_ssp245['sidmassmeltbot']['esgf'],models_gn_ssp245['sidmasslat']['esgf'],models_gn_ssp245['sidmassgrowthbot']['esgf']
           ,models_gn_ssp245['sidmassgrowthwat']['esgf'],models_gn_ssp245['sidmasssi']['esgf'],models_gn_ssp245['sidmassevapsubl']['esgf'],models_gn_ssp245['sidmassdyn']['esgf']])

In [9]:
if process_CMIP6:
    availability_desc = {var+'  '+desc: models_gn_ssp245[var] for var,desc in zip(variables,descriptions)}

    availability_df = pd.DataFrame(availability_desc).transpose().style.set_properties(**{
        'white-space': 'normal'
    })
    availability_df

In [10]:
# Redirect stderr to suppress unwanted warnings
sys.stderr = err = StringIO()
# Suppress specific Google and urllib3 loggers
for name in logging.Logger.manager.loggerDict.keys():
    if ('google' in name) or ('url' in name):
        logger = logging.getLogger(name)
        logger.setLevel(logging.CRITICAL)
        logger.propagate = False  # Prevent propagation to root logger
        for handler in logger.handlers:  # Remove existing handlers
            logger.removeHandler(handler)

## CMIP6

Load one model at a time. Currently loaded models: `ACCESS-CM2`, `HadGEM3-GC31-LL`, `UKESM1-0-LL`, `NorESM2-LM`, `NorESM2-MM`, `CESM2`, `CESM2-WACCM`, `MRI-ESM2-0`, `CNRM-CM6-1`, `CNRM-CM6-1-HR`, `CNRM-ESM2-1`, `IPSL-CM6A-LR`. Some models required corrections to standardize output to a common convention.Load one model at a time. Currently loaded models: `ACCESS-CM2`, `HadGEM3-GC31-LL`, `UKESM1-0-LL`, `NorESM2-LM`, `NorESM2-MM`, `CESM2`, `CESM2-WACCM`, `MRI-ESM2-0`, `CNRM-CM6-1`, `CNRM-CM6-1-HR`, `CNRM-ESM2-1`, `IPSL-CM6A-LR`. Some models required corrections to standardize output to a common convention.

### Load gridded budget variables

Due to the changes made to the ESGF archive and the lack of budget variables in the cloud, most of this data will be cached locally to .esgf_manual and must be deleted manually after use. Data may once again be streamable in the future after data migration stabilizes. 

In [11]:
if process_CMIP6:
    # CMIP6 variable naming: sidmass* = sea ice mass flux (kg m⁻² s⁻¹), sndmass* = snow mass flux
    CMIP6_siconc    = CMIP6(**args, variable='siconc',           source_id=models, table_id='SImon').load_data()

    CMIP6_thermo       = CMIP6(**args, variable='sidmassth',         source_id=models, table_id='SImon').load_data()
    CMIP6_basal_growth = CMIP6(**args, variable='sidmassgrowthbot',  source_id=models, table_id='SImon').load_data()
    CMIP6_frazil       = CMIP6(**args, variable='sidmassgrowthwat',  source_id=models, table_id='SImon').load_data()
    CMIP6_snow_ice     = CMIP6(**args, variable='sidmasssi',         source_id=models, table_id='SImon').load_data()
    CMIP6_top_melt     = CMIP6(**args, variable='sidmassmelttop',    source_id=models, table_id='SImon').load_data()
    CMIP6_basal_melt   = CMIP6(**args, variable='sidmassmeltbot',    source_id=models, table_id='SImon').load_data()
    CMIP6_lateral_melt = CMIP6(**args, variable='sidmasslat',        source_id=models, table_id='SImon').load_data()
    CMIP6_evap_subl    = CMIP6(**args, variable='sidmassevapsubl',   source_id=models, table_id='SImon').load_data()
    CMIP6_dynamics     = CMIP6(**args, variable='sidmassdyn',        source_id=models, table_id='SImon').load_data()

    CMIP6_snowfall      = CMIP6(**args, variable='sndmasssnf',      source_id=models, table_id='SImon').load_data()
    CMIP6_snowmelt      = CMIP6(**args, variable='sndmassmelt',     source_id=models, table_id='SImon').load_data()
    CMIP6_snow_ice_snow = CMIP6(**args, variable='sndmasssi',       source_id=models, table_id='SImon').load_data()
    CMIP6_snow_dynamics = CMIP6(**args, variable='sndmassdyn',      source_id=models, table_id='SImon').load_data()
    CMIP6_wind_drift    = CMIP6(**args, variable='sndmasswindrif',  source_id=models, table_id='SImon').load_data()
    CMIP6_evap_subl_snow = CMIP6(**args, variable='sndmasssubl',   source_id=models, table_id='SImon').load_data()

### Test variables and correct if necessary

Quick per-model sanity checks before running the full correction pipeline below: each cell plots one raw field for the first member of the currently loaded model. A model that doesn't archive a given variable returns `None` from `load_data()`, so the `try/except` silently skips the plot — an empty or missing plot here means the variable is unavailable for this model, not a bug. Obviously-wrong magnitudes or signs at this stage are what motivate the corrections documented in the `ICE_CORR`/`SNOW_CORR` dicts and the "Model notes" cells below.

In [12]:
if process_CMIP6:
    import os
    os.makedirs("figures/model_diagnostics", exist_ok=True)

In [13]:
if process_CMIP6:
    try:
        CMIP6_evap_subl_snow.sndmasssubl.isel(time=0,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmasssubl.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [14]:
if process_CMIP6:
    try:
        CMIP6_wind_drift.sndmasswindrif.isel(time=0,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmasswindrif.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [15]:
if process_CMIP6:
    try:
        CMIP6_snow_dynamics.sndmassdyn.isel(time=0,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmassdyn.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [16]:
if process_CMIP6:
    try:
        CMIP6_snow_ice_snow.sndmasssi.isel(time=0,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmasssi.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [17]:
if process_CMIP6:
    try:
        (CMIP6_snowmelt.sndmassmelt.isel(time=7,member_id=0)*180).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmassmelt.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [18]:
if process_CMIP6:
    try:
        CMIP6_snowfall.sndmasssnf.isel(time=7,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sndmasssnf.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [19]:
if process_CMIP6:
    # CNRM archives basal_growth and top_melt with overlapping/inconsistent sign conventions;
    # reconstruct them from their sum and split by sign (positive -> growth, negative -> melt).
    # Approximation: a cell with both growth and melt in the same month is misattributed entirely
    # to whichever process dominates (see "Model notes" below).
    if models[0] in ['CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1']:
        top_melt_basal_grwoth_sum = CMIP6_top_melt.sidmassmelttop+CMIP6_basal_growth.sidmassgrowthbot
        CMIP6_basal_growth['sidmassgrowthbot'] = top_melt_basal_grwoth_sum.where(lambda x:x>0).fillna(0).where(CMIP6_top_melt.mask)
        CMIP6_top_melt['sidmassmelttop'] = top_melt_basal_grwoth_sum.where(lambda x:x<0).fillna(0).where(CMIP6_top_melt.mask)

In [20]:
if process_CMIP6:
    model = models[0]
    year = 2016  # change to inspect a different year
    member=0

    # Per-model raw-data corrections
    # siconc_scale:   growth/melt/evap terms are defined per sea-ice area and need *= siconc/100
    #                 to become per-grid-cell-area.
    # siconc_exclude: term names (as used in the gt3_NH calls below) to leave un-scaled even when
    #                 siconc_scale=True — e.g. dynamics and the reported total aren't per-sea-ice-area.
    # evap_sign:      +1 if sidmassevapsubl is already negative-for-loss in the raw data, -1 if it
    #                 was published positive and needs flipping.
    # melt_sign:      +1 if top/basal/lateral melt are already negative-for-loss, -1 if they were
    #                 published positive and need flipping.
    # scale:          uniform multiplier applied to every raw ice term, on top of term_scale.
    # term_scale:     per-term multiplier (keyed by the names used in gt3_NH calls below), for when
    #                 only some variables need correcting rather than the whole model uniformly.
    ICE_CORR = {
        'ACCESS-CM2':      dict(siconc_scale=True,  siconc_exclude={'dynamics', 'thermo'}, evap_sign=-1, melt_sign=+1, scale=1, term_scale={}),
        'HadGEM3-GC31-LL': dict(siconc_scale=False, siconc_exclude=set(), evap_sign=-1, melt_sign=+1, scale=1, term_scale={}),
        'UKESM1-0-LL':     dict(siconc_scale=False, siconc_exclude=set(), evap_sign=-1, melt_sign=+1, scale=1, term_scale={}),
        'NorESM2-LM':      dict(siconc_scale=False, siconc_exclude=set(), evap_sign=+1, melt_sign=-1, scale=1, term_scale={}),
        'NorESM2-MM':      dict(siconc_scale=False, siconc_exclude=set(), evap_sign=+1, melt_sign=-1, scale=1, term_scale={}),
        'CESM2':           dict(siconc_scale=False, siconc_exclude=set(), evap_sign=+1, melt_sign=-1, scale=1, term_scale={}),
        'CESM2-WACCM':     dict(siconc_scale=False, siconc_exclude=set(), evap_sign=+1, melt_sign=-1, scale=1, term_scale={
            'thermo': 1, 'dynamics': 1, 'basal_growth': 1, 'frazil': 1, 'snow_ice': 1,
            'top_melt': 1, 'basal_melt': 1, 'lateral_melt': 1, 'evap_subl': 1/1e6,
        }),
    }
    corr = ICE_CORR.get(model)
    if corr is None:
        print(f"No documented corrections for '{model}' — assuming no siconc scaling, no sign flips, "
              "and no magnitude correction. Verify against the raw data before trusting this plot.")
        corr = dict(siconc_scale=False, siconc_exclude=set(), evap_sign=+1, melt_sign=+1, scale=1, term_scale={})
    elif model == 'CESM2-WACCM':
        print("CESM2-WACCM term_scale values were tuned empirically against this plot, not derived "
              "from documented model corrections — treat this as a debugging aid, not a validated result.")

    # sidmassth isn't archived for every model — fall back to basal_growth's grid info
    # (always present) so this diagnostic still runs; the reported-total comparison line
    # just gets skipped below when that happens.
    _areacello_src = CMIP6_thermo if CMIP6_thermo is not None else CMIP6_basal_growth
    areacello = _areacello_src.areacello
    lat = _areacello_src.lat
    siconc_da = (CMIP6_siconc.siconc.sel(member_id=_areacello_src.isel(member_id=member).member_id).sel(time=str(year)) / 100) if corr['siconc_scale'] else 1

    def gt3_NH(ds, varname, term, sign=1):
        """Area-integrate a CMIP6 sea-ice mass-flux term over the NH, member 0, in 10^3 Gt/month."""
        extra = corr['term_scale'].get(term, 1)
        siconc_factor = siconc_da if (corr['siconc_scale'] and term not in corr['siconc_exclude']) else 1
        da = ds[varname].isel(member_id=member).sel(time=str(year)) * sign * siconc_factor * corr['scale'] * extra
        seconds = da.time.dt.days_in_month * 86400
        return (da.squeeze() * areacello).where(lat > 0).sum(['x', 'y']) * seconds / 1e15

    if CMIP6_thermo is not None:
        thermo_total = gt3_NH(CMIP6_thermo, 'sidmassth', 'thermo')
    else:
        print(f"sidmassth not archived for '{model}' — skipping reported-total comparison line.")
        thermo_total = None
    basal_growth = gt3_NH(CMIP6_basal_growth, 'sidmassgrowthbot', 'basal_growth')
    frazil       = gt3_NH(CMIP6_frazil, 'sidmassgrowthwat', 'frazil')
    snow_ice     = gt3_NH(CMIP6_snow_ice, 'sidmasssi', 'snow_ice')
    top_melt     = gt3_NH(CMIP6_top_melt, 'sidmassmelttop', 'top_melt', sign=corr['melt_sign'])
    basal_melt   = gt3_NH(CMIP6_basal_melt, 'sidmassmeltbot', 'basal_melt', sign=corr['melt_sign'])
    lateral_melt_missing = False
    if CMIP6_lateral_melt is not None:
        lateral_melt = gt3_NH(CMIP6_lateral_melt, 'sidmasslat', 'lateral_melt', sign=corr['melt_sign'])
    else:
        print(f"sidmasslat not archived for '{model}' — treating lateral melt as 0 (see README Lateral Melt column).")
        lateral_melt_missing = True
        lateral_melt = xr.zeros_like(basal_melt)
    evap_subl    = gt3_NH(CMIP6_evap_subl, 'sidmassevapsubl', 'evap_subl', sign=corr['evap_sign'])
    if CMIP6_dynamics is not None:
        dynamics = gt3_NH(CMIP6_dynamics, 'sidmassdyn', 'dynamics')  # not thermodynamic — plotted for reference only
    else:
        print(f"sidmassdyn not archived for '{model}' — skipping dynamics line.")
        dynamics = None

    thermo_sum = basal_growth + frazil + snow_ice + top_melt + basal_melt + lateral_melt + evap_subl

    if lateral_melt_missing and thermo_total is not None:
        mismatch = (thermo_total - thermo_sum).sum('time')
        print(f"[diagnostic] lateral melt missing for '{model}' — reported total minus sum of "
              f"known terms = {float(mismatch):.2f} x10^3 Gt over {year} (unaccounted mass, likely "
              "includes the missing lateral melt term).")

    if model not in ICE_CORR:
        # corr's sign assumptions are unconfirmed for this model — sanity-check the resulting
        # sign of each term against the losses-negative/gains-positive convention.
        expected_negative = {'top_melt': top_melt, 'basal_melt': basal_melt, 'evap_subl': evap_subl}
        if not lateral_melt_missing:
            expected_negative['lateral_melt'] = lateral_melt
        expected_positive = {'basal_growth': basal_growth, 'frazil': frazil}
        for term_name, da in expected_negative.items():
            total = float(da.sum('time'))
            if total > 0:
                print(f"[diagnostic] {term_name} sums positive ({total:.2f} x10^3 Gt over {year}) for "
                      f"'{model}' — expected negative (loss). Raw data may need a sign flip "
                      "(set melt_sign/evap_sign=-1 for this model in ICE_CORR).")
        for term_name, da in expected_positive.items():
            total = float(da.sum('time'))
            if total < 0:
                print(f"[diagnostic] {term_name} sums negative ({total:.2f} x10^3 Gt over {year}) for "
                      f"'{model}' — expected positive (gain). Raw data may need a sign correction.")

    terms = {
        'basal growth':     (basal_growth, 'tab:blue'),
        'frazil':           (frazil, 'tab:orange'),
        'snow-to-ice':      (snow_ice, 'tab:green'),
        'top melt':         (top_melt, 'tab:red'),
        'basal melt':       (basal_melt, 'tab:purple'),
        'lateral melt':     (lateral_melt, 'tab:brown'),
        'evap/sublimation': (evap_subl, 'tab:pink'),
    }
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, (da, color) in terms.items():
        ax.plot(da.time, da, color=color, lw=1.3, label=label)
    if dynamics is not None:
        ax.plot(dynamics.time, dynamics, color='0.5', lw=1.3, ls=':', label='dynamics (excluded from sum)')
    ax.plot(thermo_sum.time, thermo_sum, color='r', lw=4, ls=':', label='sum of thermo terms')
    if thermo_total is not None:
        ax.plot(thermo_total.time, thermo_total, color='k',ls='--', lw=2.5, label='reported total (sidmassth)')
    ax.axhline(0, color='0.7', lw=0.8)
    #ax.set_ylim(-1,1)
    ax.set_ylabel(r'$10^3$ Gt month$^{-1}$')
    ax.set_title(f'{model} — sea ice thermodynamic mass budget, NH {year}')
    ax.legend(ncol=2, fontsize=8, loc='best')
    plt.tight_layout()
    plt.show()

In [21]:
if process_CMIP6:
    model = models[0]
    year = 2015  # keep in sync with the ice-budget cell above
    member = 0
    SNOW_CORR = {
        'ACCESS-CM2':      dict(siconc_scale=True,  siconc_exclude=set(), melt_sign=+1, scale=1, term_scale={'snowmelt': 330/1000}),
        'HadGEM3-GC31-LL': dict(siconc_scale=False, siconc_exclude=set(), melt_sign=+1, scale=1, term_scale={}),
        'UKESM1-0-LL':     dict(siconc_scale=False, siconc_exclude=set(), melt_sign=+1, scale=1, term_scale={}),
        'NorESM2-LM':      dict(siconc_scale=False, siconc_exclude=set(), melt_sign=-1, scale=1, term_scale={'snowfall': 1/330}),
        'NorESM2-MM':      dict(siconc_scale=False, siconc_exclude=set(), melt_sign=-1, scale=1, term_scale={'snowfall': 1/330}),
        'CESM2':           dict(siconc_scale=False, siconc_exclude=set(), melt_sign=-1, scale=1, term_scale={}),
        'CESM2-LE':        dict(siconc_scale=False, siconc_exclude=set(), melt_sign=-1, scale=1, term_scale={}),
        'CESM2-WACCM':     dict(siconc_scale=False, siconc_exclude=set(), melt_sign=-1, scale=1, term_scale={'snowmelt': 1}),
    }
    scorr = SNOW_CORR.get(model)
    if scorr is None:
        print(f"No documented corrections for '{model}' — assuming no siconc scaling, no sign flip, "
              "and no magnitude correction. Verify against the raw data before trusting this plot.")
        scorr = dict(siconc_scale=False, siconc_exclude=set(), melt_sign=+1, scale=1, term_scale={})
    
    areacello = CMIP6_siconc.areacello
    lat = CMIP6_siconc.lat
    siconc_da = (CMIP6_siconc.siconc.sel(member_id=CMIP6_snowfall.isel(member_id=member).member_id).sel(time=str(year)) / 100) if scorr['siconc_scale'] else 1

    def gt3_NH_snow(ds, varname, sign=1, term=None):
        """Area-integrate a CMIP6 snow mass-flux term over the NH, member 0, in 10^3 Gt/month."""
        extra = scorr['term_scale'].get(term, 1) if term else 1
        siconc_factor = siconc_da if (scorr['siconc_scale'] and term not in scorr['siconc_exclude']) else 1
        da = ds[varname].isel(member_id=member).sel(time=str(year)) * sign * siconc_factor * scorr['scale'] * extra
        seconds = da.time.dt.days_in_month * 86400
        return (da.squeeze() * areacello).where(lat > 0).sum(['x', 'y']) * seconds / 1e15

    # (dataset, variable, sign, color) — not every model publishes every one of these
    snow_candidates = {
        'snowfall':         (CMIP6_snowfall,       'sndmasssnf',     1,                  'tab:blue'),
        'snowmelt':         (CMIP6_snowmelt,       'sndmassmelt',    scorr['melt_sign'], 'tab:red'),
        'snow-to-ice':      (CMIP6_snow_ice_snow,  'sndmasssi',      1,                  'tab:green'),
        'wind drift':       (CMIP6_wind_drift,     'sndmasswindrif', 1,                  'tab:orange'),
        'evap/sublimation': (CMIP6_evap_subl_snow, 'sndmasssubl',    1,                  'tab:pink'),
        'dynamics':         (CMIP6_snow_dynamics,  'sndmassdyn',     1,                  'tab:purple'),
    }

    snow_terms = {}
    for label, (ds, varname, sign, color) in snow_candidates.items():
        try:
            snow_terms[label] = (gt3_NH_snow(ds, varname, sign=sign, term=label), color)
        except Exception as e:
            print(f"Skipping '{label}' for {model}: {type(e).__name__} ({e})")

    fig, ax = plt.subplots(figsize=(9, 5))
    for label, (da, color) in snow_terms.items():
        ax.plot(da.time, da, color=color, lw=1.3, label=label)
    if snow_terms:
        snow_sum = sum(da for da, _ in snow_terms.values())
        ax.plot(snow_sum.time, snow_sum, color='k', lw=2.5, ls='--', label='sum of available terms')
    ax.axhline(0, color='0.7', lw=0.8)
    ax.set_ylabel(r'$10^3$ Gt month$^{-1}$')
    ax.set_title(f'{model} — snow mass budget (available terms only), NH {year}')
    ax.legend(ncol=2, fontsize=8, loc='best')
    plt.tight_layout()
    plt.show()

In [22]:
if process_CMIP6:
    CMIP6_top_melt.sidmassmelttop.isel(time=6,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassmelttop.png", dpi=150, bbox_inches='tight')

In [23]:
if process_CMIP6:
    CMIP6_basal_growth.sidmassgrowthbot.isel(time=6,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassgrowthbot.png", dpi=150, bbox_inches='tight')

In [24]:
if process_CMIP6:
    try:
        CMIP6_evap_subl.sidmassevapsubl.isel(time=0,member_id=0).plot()
        plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassevapsubl.png", dpi=150, bbox_inches='tight')
    except:
        pass

In [25]:
if process_CMIP6:
    CMIP6_dynamics.sidmassdyn.isel(member_id=0).sel(time='2015').mean('time').plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassdyn.png", dpi=150, bbox_inches='tight')

In [26]:
if process_CMIP6:
    CMIP6_frazil.sidmassgrowthwat.isel(time=7,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassgrowthwat.png", dpi=150, bbox_inches='tight')

In [27]:
if process_CMIP6:
    CMIP6_snow_ice.sidmasssi.isel(time=8,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmasssi.png", dpi=150, bbox_inches='tight')

In [28]:
if process_CMIP6:
    if CMIP6_lateral_melt is not None:
        _lm_slice = CMIP6_lateral_melt.sidmasslat.isel(time=0,member_id=0)
        if bool((_lm_slice == 0).all()):
            print(f"sidmasslat is archived for {models[0]} but this slice (time=0, member=0) is "
                  "identically zero \u2014 likely no genuine output; skipping diagnostic plot rather "
                  "than showing a blank/misleading map.")
        else:
            _lm_slice.plot()
            plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmasslat.png", dpi=150, bbox_inches='tight')
    else:
        print(f"sidmasslat not available for {models[0]} — skipping diagnostic plot.")

In [29]:
if process_CMIP6:
    CMIP6_thermo.sidmassth.isel(time=0,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassth.png", dpi=150, bbox_inches='tight')

In [30]:
if process_CMIP6:
    CMIP6_basal_melt.sidmassmeltbot.isel(time=0,member_id=0).plot()
    plt.savefig(f"figures/model_diagnostics/{models[0]}_sidmassmeltbot.png", dpi=150, bbox_inches='tight')

Cross-check: `sidmasssi` (ice-side snow-to-ice conversion, from the ice budget) and `sndmasssi` (snow-side, from the snow budget) should be related by the snow/ice density ratio under Archimedean flooding. Inverting that relation from their climatological ratio `R` gives an implied snow density — comparing it to the ~330 kg m$^{-3}$ used in the ACCESS-CM2 and NorESM2 corrections is a sanity check that those density-based fixes are physically reasonable, not just tuned to look right.

In [31]:
if process_CMIP6:
    RHO_W, RHO_I = 1026.0, 917.0  

    def rho_snow_from_R(R, rho_w=RHO_W, rho_i=RHO_I):
        """Invert R = 1 + rho_w/rho_s - rho_w/rho_i (Archimedean flooding)."""
        return rho_w / (R - 1.0 + rho_w / rho_i)

    def hemi_integral(da, mask, member=0):
        """Flux (kg m-2 s-1) -> Gt month-1, integrated over mask."""
        sec = da.time.dt.days_in_month * 86400
        return ((da.isel(member_id=member) * CMIP6_thermo.areacello)
                .where(mask).sum(['x', 'y']) * sec / 1e12)

    MASK = CMIP6_snow_ice.lat<0  # <- swap in whatever the panel figure used

In [32]:
if process_CMIP6:
    si_clim = (hemi_integral(CMIP6_snow_ice.sidmasssi, MASK)
               .sel(time=slice('2015', '2034')).groupby('time.month').mean()).squeeze()
    sn_clim = (-hemi_integral(CMIP6_snow_ice_snow.sndmasssi, MASK)
               .sel(time=slice('2015', '2034')).groupby('time.month').mean()).squeeze()

    R = si_clim / sn_clim

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].plot(si_clim.month, si_clim, 'r-',  lw=2, label='sidmasssi')
    ax[0].plot(sn_clim.month, sn_clim, 'r--', lw=2, label='|sndmasssi|')
    ax[0].set_ylabel('Gt month$^{-1}$'); ax[0].legend(); ax[0].set_title('snow-ice, both budgets')

    ax[1].plot(R.month, R, 'k-o', ms=4)
    for r, ls, lab in [(2.779, ':', '917/330'),
                       (1 + RHO_W/330 - RHO_W/RHO_I, '--', 'flooding, $\\rho_s$=330')]:
        ax[1].axhline(r, color='0.5', ls=ls, label=lab)
    ax[1].set_ylabel('R'); ax[1].legend(fontsize=8); ax[1].set_title('mass ratio')

    ax[2].plot(R.month, rho_snow_from_R(R), 'k-o', ms=4)
    ax[2].set_ylabel('inferred $\\rho_s$ (kg m$^{-3}$)'); ax[2].set_title('implied snow density')
    for a in ax: a.set_xlabel('Month')
    plt.tight_layout()
    fig.savefig(f"figures/model_diagnostics/{models[0]}_snow_ice_density_check.png", dpi=150, bbox_inches='tight')

### Area-integrate snow fluxes (10³ Gt month⁻¹)

 For the inner Arctic Ocean domain, we use Arctic Ocean basin (central Arctic plus the Beaufort, Chukchi, East Siberian, Laptev and Kara seas) and the Barents Sea.
 This is [0,1,2,3,4,5,6,7] in `NH_seaice_regions` from the NSIDC

In [33]:
if process_CMIP6:
    NH_seaice_regions

Model notes:

- ACCESS-CM2:
    - All variables are defined per sea ice area and were converted to grid cell area
    - It is not a documented error, but the snowmelt appears much larger than physically possible. Snowmelt should use snow density (330) for CICE cm/day -> kg/s, but appears to have erroneously used freshwater density (1000) instead
    - sndmasswindrif and sndmasssubl are unavailable
- HadGEM3-GC31-LL and UKESM1-0-LL
    - sndmasswindrif and sndmasssubl are unavailable
- NorESM-LM and MM
    - Snowmelt was defined positive and switched to a negative flux.
    - Snowfall also need to be divided by snow density (330).
    - sndmasssubl and sndmassdyn are unavailable
- CESM2
    - Snowmelt was defined positive and switched to a negative flux.
- CESM2-WACCM
    - Snowmelt was defined positive and switched to a negative flux.
    - Correction is per-member, not model-wide: only the first member (r1i1p1f1) has archived snowmelt that is 1800x too small (same 30-minute/1800s coupling-interval issue as the ice-budget terms below); other members are archived correctly and are left unscaled.
    - Only the third member (r3i1p1f1) has archived snowfall that is ~330x too large (same snow-density-unit issue as NorESM2's fix); other members' snowfall is archived correctly.
    - sndmasssubl (unlike other models) is available for CESM2-WACCM, but is left unscaled since its magnitude hasn't been checked. Snow-to-ice and wind drift also not yet checked.
- MRI-ESM2-0, CNRM-CM6-1, CNRM-CM6-1-HR, CNRM-ESM2-1, and IPSL-CM6A-LR
    - No corrections needed.
    - sndmasswindrif is unavailable

In [34]:
if process_CMIP6:
    seconds = CMIP6_siconc.time.dt.days_in_month * 86400

    if models[0] == 'ACCESS-CM2':
        conversion_factor = 1000/330
        # TODO: unconfirmed, this my not be the accurate correction
        CMIP6_snowmelt_SH      = (((CMIP6_snowmelt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/conversion_factor
        CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/conversion_factor
        CMIP6_snowmelt_NH      = (((CMIP6_snowmelt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/conversion_factor
        CMIP6_snowmelt_IA      = ((region_mask(CMIP6_snowmelt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt/conversion_factor
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
        CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_IA      = ((region_mask(CMIP6_snow_ice_snow*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
        CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_IA      = ((region_mask(CMIP6_snow_dynamics*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn

    if models[0] in ['HadGEM3-GC31-LL','UKESM1-0-LL']:
        CMIP6_snowmelt_SH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = ((region_mask(CMIP6_snowmelt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
        CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_IA      = ((region_mask(CMIP6_snow_ice_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
        CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_IA      = ((region_mask(CMIP6_snow_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn

    if models[0] in ['NorESM2-LM','NorESM2-MM']:

        CMIP6_snowmelt_SH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = -((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = -((region_mask(CMIP6_snowmelt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330
        CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf/330

        CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_IA      = ((region_mask(CMIP6_snow_ice_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
        CMIP6_wind_drift_SH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
        CMIP6_wind_drift_Weddell = ((region_mask(CMIP6_wind_drift,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
        CMIP6_wind_drift_NH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
        CMIP6_wind_drift_IA      = ((region_mask(CMIP6_wind_drift,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif

    if models[0] in ['CESM2']:
        CMIP6_snowmelt_SH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = -((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = -(((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = -((region_mask(CMIP6_snowmelt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf

    if models[0] == 'CESM2-WACCM':
        # Only the first ensemble member (r1i1p1f1) has archived snowmelt that is 1800x too
        # small (matches the ice budget's 30-minute/1800s coupling-interval bug below); the
        # other members are archived correctly and must not be rescaled. Separately, only the
        # third member (r3i1p1f1) has archived snowfall that is ~330x too large (same snow-
        # density-unit issue as NorESM2's snowfall/330 fix); the other members' snowfall is
        # archived correctly. sndmasssubl (unlike other models) is available for CESM2-WACCM;
        # unscaled since its magnitude hasn't been checked. snow-to-ice and wind drift not yet
        # checked.
        waccm_member0 = CMIP6_snowmelt.member_id == 'CESM2-WACCM_r1i1p1f1'
        waccm_member2 = CMIP6_snowfall.member_id == 'CESM2-WACCM_r2i1p1f1'
        snowmelt_scale = xr.where(waccm_member0, 1800, 1)
        snowfall_scale = xr.where(waccm_member2, 1/330, 1)

        CMIP6_snowmelt_SH      = -(((CMIP6_snowmelt*snowmelt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = -((region_mask(CMIP6_snowmelt*snowmelt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = -(((CMIP6_snowmelt*snowmelt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = -((region_mask(CMIP6_snowmelt*snowmelt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall*snowfall_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall*snowfall_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall*snowfall_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall*snowfall_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
        CMIP6_evap_subl_snow_SH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_Weddell = ((region_mask(CMIP6_evap_subl_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_NH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_IA      = ((region_mask(CMIP6_evap_subl_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl

    if models[0] in ['MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR','CNRM-ESM2-1']:
        # No corrections needed: archived terms already follow the CF sign convention
        # (negative-for-loss) with no unit/magnitude issues. sndmasswindrif is unavailable.
        CMIP6_snowmelt_SH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = ((region_mask(CMIP6_snowmelt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
    
        CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
    
        CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
        CMIP6_snow_ice_snow_IA      = ((region_mask(CMIP6_snow_ice_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
    
        CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
        CMIP6_snow_dynamics_IA      = ((region_mask(CMIP6_snow_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
    
        CMIP6_evap_subl_snow_SH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_Weddell = ((region_mask(CMIP6_evap_subl_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_NH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
        CMIP6_evap_subl_snow_IA      = ((region_mask(CMIP6_evap_subl_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl

    if models[0] not in ['ACCESS-CM2', 'HadGEM3-GC31-LL', 'UKESM1-0-LL', 'NorESM2-LM', 'NorESM2-MM', 'CESM2', 'CESM2-WACCM', 'MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1']:
        print(f"No documented snow corrections for '{models[0]}' — assuming no siconc scaling and no "
              "sign flips. Verify against the raw data before trusting this plot.")
        CMIP6_snowmelt_SH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_Weddell = ((region_mask(CMIP6_snowmelt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_NH      = (((CMIP6_snowmelt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt
        CMIP6_snowmelt_IA      = ((region_mask(CMIP6_snowmelt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassmelt

        CMIP6_snowfall_SH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_Weddell = ((region_mask(CMIP6_snowfall,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_NH      = (((CMIP6_snowfall).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf
        CMIP6_snowfall_IA      = ((region_mask(CMIP6_snowfall,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssnf

        if CMIP6_snow_ice_snow is not None:
            CMIP6_snow_ice_snow_SH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
            CMIP6_snow_ice_snow_Weddell = ((region_mask(CMIP6_snow_ice_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
            CMIP6_snow_ice_snow_NH      = (((CMIP6_snow_ice_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi
            CMIP6_snow_ice_snow_IA      = ((region_mask(CMIP6_snow_ice_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssi

        if CMIP6_snow_dynamics is not None:
            CMIP6_snow_dynamics_SH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
            CMIP6_snow_dynamics_Weddell = ((region_mask(CMIP6_snow_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
            CMIP6_snow_dynamics_NH      = (((CMIP6_snow_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn
            CMIP6_snow_dynamics_IA      = ((region_mask(CMIP6_snow_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmassdyn

        if CMIP6_wind_drift is not None:
            CMIP6_wind_drift_SH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
            CMIP6_wind_drift_Weddell = ((region_mask(CMIP6_wind_drift,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
            CMIP6_wind_drift_NH      = (((CMIP6_wind_drift).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif
            CMIP6_wind_drift_IA      = ((region_mask(CMIP6_wind_drift,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasswindrif

        if CMIP6_evap_subl_snow is not None:
            CMIP6_evap_subl_snow_SH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
            CMIP6_evap_subl_snow_Weddell = ((region_mask(CMIP6_evap_subl_snow,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
            CMIP6_evap_subl_snow_NH      = (((CMIP6_evap_subl_snow).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl
            CMIP6_evap_subl_snow_IA      = ((region_mask(CMIP6_evap_subl_snow,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sndmasssubl


### Area-integrate ice fluxes (10³ Gt month⁻¹)

- ACCESS-CM2:
    - Thermodynamics and dynamics are defined relative to grid cell area
    - All other variables are incorrectly defined relative to sea ice area and need to be scaled by `siconc/100` before computing the spatial sum.
    - sidmassevapsubl (evaporation and sublimation) was defined positive in the model output and has been changed to negative
- HadGEM3-GC31-LL and UKESM1-0-LL
    - sidmassevapsubl (evaporation and sublimation) was defined positive in the model output and has been changed to negative
- NorESM-LM, MM, and CESM2
    - Melt terms (top melt: sidmassmelttop, basal melt: sidmassmeltbot, and lateral melt: sidmasslat) were defined positive in the model output and have been changed to negative
- CESM2-WACCM
    - Correction is per-member, not model-wide. Only the first member (r1i1p1f1) has basal_growth, frazil, snow_ice, top_melt, basal_melt, and lateral_melt archived 1800x too small (matches CESM2's 30-minute/1800s ice-ocean coupling interval, though the root cause in the diagnostic pipeline hasn't been confirmed); other members are archived correctly.
    - Conversely, evap_subl is 1e6x too large in every member except the first, which is already correct as archived.
    - thermo and dynamics did not need correction in any member.
- MRI-ESM2-0 and IPSL-CM6A-LR
    - No corrections needed. `sidmasslat` (lateral melt) is not archived for IPSL-CM6A-LR — treated as 0 rather than raising.
- CNRM-CM6-1, CNRM-CM6-1-HR, and CNRM-ESM2-1
    - basal_growth (sidmassgrowthbot) and top_melt (sidmassmelttop) are reconstructed from their sum: the two raw terms are added together, then the combined field is split by sign — positive values are assigned to basal growth, negative values to top melt (zero elsewhere, masked to the original footprint). This is an approximation: a grid cell with both growth and melt occurring in the same month is misattributed entirely to whichever process dominates.
    - All other ice terms: no corrections needed.

In [35]:
def _lateral_melt_or_missing(CMIP6_lateral_melt, model_name):
    """Area-integrate sidmasslat over SH/NH/Weddell/IA, or return four Nones if the
    variable isn't archived, or is archived but identically zero for every member
    (e.g. MRI-ESM2-0) — a real flux never sums to exactly zero, so treat that the
    same as "not archived" rather than diluting the multi-model mean with it."""
    if CMIP6_lateral_melt is None:
        print(f"sidmasslat not archived for '{model_name}' — treating lateral melt as missing.")
        return None, None, None, None

    lm_SH = (((CMIP6_lateral_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    lm_NH = (((CMIP6_lateral_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    lm_Weddell = ((region_mask(CMIP6_lateral_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    lm_IA = ((region_mask(CMIP6_lateral_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat

    if bool((lm_SH == 0).all()) and bool((lm_NH == 0).all()):
        print(f"sidmasslat is archived for '{model_name}' but is identically zero — treating as missing.")
        return None, None, None, None

    return lm_SH, lm_NH, lm_Weddell, lm_IA


In [36]:
if process_CMIP6:
    if models[0] == 'ACCESS-CM2':

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = ((region_mask(CMIP6_lateral_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = ((region_mask(CMIP6_lateral_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] in ['HadGEM3-GC31-LL','UKESM1-0-LL']:

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])
    
        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] in ['NorESM2-LM','NorESM2-MM','CESM2']:

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = -((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = -((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = -((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = -((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = -((region_mask(CMIP6_lateral_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = -((region_mask(CMIP6_lateral_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] == 'CESM2-WACCM':
        # Only the first ensemble member (r1i1p1f1) has archived basal_growth/frazil/snow_ice/
        # top_melt/basal_melt/lateral_melt terms that are 1800x too small (matches CESM2's
        # 30-minute/1800s ice-ocean coupling interval, though the root cause in the diagnostic
        # pipeline hasn't been confirmed); the other members are archived correctly. Conversely,
        # evap_subl is 1e6x too large in every member except the first, which is already correct
        # as archived. thermo and dynamics did not need correction in any member.
        waccm_member0 = CMIP6_thermo.member_id == 'CESM2-WACCM_r1i1p1f1'
        growth_melt_scale = xr.where(waccm_member0, 1800, 1)
        evap_subl_scale = xr.where(waccm_member0, 1, 1/1e6)

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = -(((CMIP6_top_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = -(((CMIP6_top_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = -((region_mask(CMIP6_top_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = -((region_mask(CMIP6_top_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = -(((CMIP6_basal_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = -(((CMIP6_basal_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = -((region_mask(CMIP6_basal_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = -((region_mask(CMIP6_basal_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = -(((CMIP6_lateral_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = -(((CMIP6_lateral_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = -((region_mask(CMIP6_lateral_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = -((region_mask(CMIP6_lateral_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl*evap_subl_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl*evap_subl_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl*evap_subl_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl*evap_subl_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn

    if models[0] in ['MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR','CNRM-ESM2-1', 'IPSL-CM6A-LR', 'EC-Earth3']:
        # No corrections needed: archived terms already follow the CF sign convention
        # (negative-for-loss) with no unit/magnitude issues.
        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        if models[0] in ['IPSL-CM6A-LR', 'MRI-ESM2-0']:
            # sidmassdyn is anomalous for these models: hemispheric (SH/NH) dynamics should be a
            # near-zero closed-domain residual (every other model here is within ~150 Gt/yr), but
            # IPSL's NH residual is ~1500 Gt/yr and MRI-ESM2-0's is far larger still (~2700 Gt/yr
            # SH, ~720 Gt/yr NH, ~1000–1350 Gt/yr IA), consistent across all members and never
            # changing sign. Confirmed present in the raw sidmassdyn field itself (computed
            # directly on each model's native grid, bypassing this pipeline's masking/area-
            # weighting entirely), so it's not a processing bug here — root cause unconfirmed
            # (see README Known Issues). Treated as missing for all regions rather than plotting
            # a term we don't trust.
            print(f"sidmassdyn for '{models[0]}' is anomalous (see README Known Issues) \u2014 treating dynamics as missing.")
            CMIP6_dynamics_SH = CMIP6_dynamics_NH = CMIP6_dynamics_Weddell = CMIP6_dynamics_IA = None
        else:
            CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn

    if models[0] not in ['ACCESS-CM2', 'HadGEM3-GC31-LL', 'UKESM1-0-LL', 'NorESM2-LM', 'NorESM2-MM', 'CESM2', 'CESM2-WACCM', 'MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'IPSL-CM6A-LR', 'EC-Earth3']:
        print(f"No documented ice corrections for '{models[0]}' — assuming no siconc scaling and no "
              "sign flips. Verify against the raw data before trusting this plot.")
        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth

        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot

        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat

        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi

        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop

        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot

        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])

        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl

        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
if process_CMIP6:
    if models[0] == 'ACCESS-CM2':

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = (((CMIP6_lateral_melt*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = ((region_mask(CMIP6_lateral_melt*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = ((region_mask(CMIP6_lateral_melt*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl*(CMIP6_siconc.siconc/100)).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl*(CMIP6_siconc.siconc/100),{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl*(CMIP6_siconc.siconc/100),{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] in ['HadGEM3-GC31-LL','UKESM1-0-LL']:

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])
    
        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] in ['NorESM2-LM','NorESM2-MM','CESM2']:

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = -(((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = -((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = -((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = -(((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = -((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = -((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = -(((CMIP6_lateral_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = -((region_mask(CMIP6_lateral_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = -((region_mask(CMIP6_lateral_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
    
    if models[0] == 'CESM2-WACCM':
        # Only the first ensemble member (r1i1p1f1) has archived basal_growth/frazil/snow_ice/
        # top_melt/basal_melt/lateral_melt terms that are 1800x too small (matches CESM2's
        # 30-minute/1800s ice-ocean coupling interval, though the root cause in the diagnostic
        # pipeline hasn't been confirmed); the other members are archived correctly. Conversely,
        # evap_subl is 1e6x too large in every member except the first, which is already correct
        # as archived. thermo and dynamics did not need correction in any member.
        waccm_member0 = CMIP6_thermo.member_id == 'CESM2-WACCM_r1i1p1f1'
        growth_melt_scale = xr.where(waccm_member0, 1800, 1)
        evap_subl_scale = xr.where(waccm_member0, 1, 1/1e6)

        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = -(((CMIP6_top_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = -(((CMIP6_top_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = -((region_mask(CMIP6_top_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = -((region_mask(CMIP6_top_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = -(((CMIP6_basal_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = -(((CMIP6_basal_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = -((region_mask(CMIP6_basal_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = -((region_mask(CMIP6_basal_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH = -(((CMIP6_lateral_melt*growth_melt_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_NH = -(((CMIP6_lateral_melt*growth_melt_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_Weddell = -((region_mask(CMIP6_lateral_melt*growth_melt_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
        CMIP6_lateral_melt_IA = -((region_mask(CMIP6_lateral_melt*growth_melt_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasslat
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl*evap_subl_scale).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl*evap_subl_scale).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl*evap_subl_scale,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl*evap_subl_scale,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn

    if models[0] in ['MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR','CNRM-ESM2-1', 'IPSL-CM6A-LR', 'EC-Earth3']:
        # No corrections needed: archived terms already follow the CF sign convention
        # (negative-for-loss) with no unit/magnitude issues.
        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
    
        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
    
        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
    
        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
    
        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
    
        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
    
        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])
    
        CMIP6_evap_subl_SH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = (((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = ((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = ((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
    
        if models[0] in ['IPSL-CM6A-LR', 'MRI-ESM2-0']:
            # sidmassdyn is anomalous for these models: hemispheric (SH/NH) dynamics should be a
            # near-zero closed-domain residual (every other model here is within ~150 Gt/yr), but
            # IPSL's NH residual is ~1500 Gt/yr and MRI-ESM2-0's is far larger still (~2700 Gt/yr
            # SH, ~720 Gt/yr NH, ~1000–1350 Gt/yr IA), consistent across all members and never
            # changing sign. Confirmed present in the raw sidmassdyn field itself (computed
            # directly on each model's native grid, bypassing this pipeline's masking/area-
            # weighting entirely), so it's not a processing bug here — root cause unconfirmed
            # (see README Known Issues). Treated as missing for all regions rather than plotting
            # a term we don't trust.
            print(f"sidmassdyn for '{models[0]}' is anomalous (see README Known Issues) \u2014 treating dynamics as missing.")
            CMIP6_dynamics_SH = CMIP6_dynamics_NH = CMIP6_dynamics_Weddell = CMIP6_dynamics_IA = None
        else:
            CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
            CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn

    if models[0] not in ['ACCESS-CM2', 'HadGEM3-GC31-LL', 'UKESM1-0-LL', 'NorESM2-LM', 'NorESM2-MM', 'CESM2', 'CESM2-WACCM', 'MRI-ESM2-0', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'IPSL-CM6A-LR', 'EC-Earth3']:
        print(f"No documented ice corrections for '{models[0]}' — assuming no siconc scaling and no "
              "sign flips. Verify against the raw data before trusting this plot.")
        CMIP6_thermo_SH = (((CMIP6_thermo).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_NH = (((CMIP6_thermo).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_Weddell = ((region_mask(CMIP6_thermo,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth
        CMIP6_thermo_IA = ((region_mask(CMIP6_thermo,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassth

        CMIP6_basal_growth_SH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_NH = (((CMIP6_basal_growth).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_Weddell = ((region_mask(CMIP6_basal_growth,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot
        CMIP6_basal_growth_IA = ((region_mask(CMIP6_basal_growth,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthbot

        CMIP6_frazil_SH = (((CMIP6_frazil).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_NH = (((CMIP6_frazil).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_Weddell = ((region_mask(CMIP6_frazil,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat
        CMIP6_frazil_IA = ((region_mask(CMIP6_frazil,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassgrowthwat

        CMIP6_snow_ice_SH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_NH = (((CMIP6_snow_ice).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_Weddell = ((region_mask(CMIP6_snow_ice,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi
        CMIP6_snow_ice_IA = ((region_mask(CMIP6_snow_ice,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmasssi

        CMIP6_top_melt_SH = (((CMIP6_top_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_NH = (((CMIP6_top_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_Weddell = ((region_mask(CMIP6_top_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop
        CMIP6_top_melt_IA = ((region_mask(CMIP6_top_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmelttop

        CMIP6_basal_melt_SH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_NH = (((CMIP6_basal_melt).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_Weddell = ((region_mask(CMIP6_basal_melt,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot
        CMIP6_basal_melt_IA = ((region_mask(CMIP6_basal_melt,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassmeltbot

        CMIP6_lateral_melt_SH, CMIP6_lateral_melt_NH, CMIP6_lateral_melt_Weddell, CMIP6_lateral_melt_IA = \
            _lateral_melt_or_missing(CMIP6_lateral_melt, models[0])

        CMIP6_evap_subl_SH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_NH = -(((CMIP6_evap_subl).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_Weddell = -((region_mask(CMIP6_evap_subl,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl
        CMIP6_evap_subl_IA = -((region_mask(CMIP6_evap_subl,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassevapsubl

        CMIP6_dynamics_SH = (((CMIP6_dynamics).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_NH = (((CMIP6_dynamics).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_Weddell = ((region_mask(CMIP6_dynamics,{'Antarctic':[0]}).where(CMIP6_siconc.lat<0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn
        CMIP6_dynamics_IA = ((region_mask(CMIP6_dynamics,{'Arctic': [0,1,2,3,4,5,6,7]}).where(CMIP6_siconc.lat>0)*CMIP6_siconc.areacello).sum(['x','y'])*seconds/1e15).sidmassdyn


### Merge and save

In [37]:
if process_CMIP6:
    CMIP6_snowfall_ds = xr.merge([CMIP6_snowfall_SH.to_dataset(name='snowfall_SH'),
                            CMIP6_snowfall_NH.to_dataset(name='snowfall_NH'),
                            CMIP6_snowfall_IA.to_dataset(name='snowfall_IA'),
                            CMIP6_snowfall_Weddell.to_dataset(name='snowfall_Weddell')])
    CMIP6_snowmelt_ds = xr.merge([CMIP6_snowmelt_SH.to_dataset(name='snowmelt_SH'),
                            CMIP6_snowmelt_NH.to_dataset(name='snowmelt_NH'),
                            CMIP6_snowmelt_IA.to_dataset(name='snowmelt_IA'),
                            CMIP6_snowmelt_Weddell.to_dataset(name='snowmelt_Weddell')])
    if CMIP6_snow_ice_snow is not None:
        CMIP6_snow_ice_snow_ds = xr.merge([CMIP6_snow_ice_snow_SH.to_dataset(name='snow_ice_snow_SH'),
                                 CMIP6_snow_ice_snow_NH.to_dataset(name='snow_ice_snow_NH'),
                                 CMIP6_snow_ice_snow_IA.to_dataset(name='snow_ice_snow_IA'),
                                 CMIP6_snow_ice_snow_Weddell.to_dataset(name='snow_ice_snow_Weddell')])
    if CMIP6_snow_dynamics is not None:
        CMIP6_snow_dynamics_ds = xr.merge([CMIP6_snow_dynamics_SH.to_dataset(name='snow_dynamics_SH'),
                                     CMIP6_snow_dynamics_NH.to_dataset(name='snow_dynamics_NH'),
                                     CMIP6_snow_dynamics_IA.to_dataset(name='snow_dynamics_IA'),
                                     CMIP6_snow_dynamics_Weddell.to_dataset(name='snow_dynamics_Weddell')])
    if CMIP6_wind_drift is not None:
        CMIP6_wind_drift_ds = xr.merge([CMIP6_wind_drift_SH.to_dataset(name='wind_drift_SH'),
                                  CMIP6_wind_drift_NH.to_dataset(name='wind_drift_NH'),
                                  CMIP6_wind_drift_IA.to_dataset(name='wind_drift_IA'),
                                  CMIP6_wind_drift_Weddell.to_dataset(name='wind_drift_Weddell')])
    if CMIP6_evap_subl_snow is not None:
        CMIP6_evap_subl_snow_ds = xr.merge([CMIP6_evap_subl_snow_SH.to_dataset(name='evap_subl_snow_SH'),
                                      CMIP6_evap_subl_snow_NH.to_dataset(name='evap_subl_snow_NH'),
                                      CMIP6_evap_subl_snow_IA.to_dataset(name='evap_subl_snow_IA'),
                                      CMIP6_evap_subl_snow_Weddell.to_dataset(name='evap_subl_snow_Weddell')])

    CMIP6_thermo_ds = xr.merge([CMIP6_thermo_SH.to_dataset(name='thermo_SH')
              ,CMIP6_thermo_NH.to_dataset(name='thermo_NH')
              ,CMIP6_thermo_IA.to_dataset(name='thermo_IA')
              ,CMIP6_thermo_Weddell.to_dataset(name='thermo_Weddell')])
    CMIP6_basal_growth_ds = xr.merge([CMIP6_basal_growth_SH.to_dataset(name='basal_growth_SH')
              ,CMIP6_basal_growth_NH.to_dataset(name='basal_growth_NH')
              ,CMIP6_basal_growth_IA.to_dataset(name='basal_growth_IA')
              ,CMIP6_basal_growth_Weddell.to_dataset(name='basal_growth_Weddell')])
    CMIP6_frazil_ds = xr.merge([CMIP6_frazil_SH.to_dataset(name='frazil_SH')
              ,CMIP6_frazil_NH.to_dataset(name='frazil_NH')
              ,CMIP6_frazil_IA.to_dataset(name='frazil_IA')
              ,CMIP6_frazil_Weddell.to_dataset(name='frazil_Weddell')])
    CMIP6_snow_ice_ds = xr.merge([CMIP6_snow_ice_SH.to_dataset(name='snow_ice_SH')
              ,CMIP6_snow_ice_NH.to_dataset(name='snow_ice_NH')
              ,CMIP6_snow_ice_IA.to_dataset(name='snow_ice_IA')
              ,CMIP6_snow_ice_Weddell.to_dataset(name='snow_ice_Weddell')])
    CMIP6_top_melt_ds = xr.merge([CMIP6_top_melt_SH.to_dataset(name='top_melt_SH')
              ,CMIP6_top_melt_NH.to_dataset(name='top_melt_NH')
              ,CMIP6_top_melt_IA.to_dataset(name='top_melt_IA')
              ,CMIP6_top_melt_Weddell.to_dataset(name='top_melt_Weddell')])
    CMIP6_basal_melt_ds = xr.merge([CMIP6_basal_melt_SH.to_dataset(name='basal_melt_SH')
              ,CMIP6_basal_melt_NH.to_dataset(name='basal_melt_NH')
              ,CMIP6_basal_melt_IA.to_dataset(name='basal_melt_IA')
              ,CMIP6_basal_melt_Weddell.to_dataset(name='basal_melt_Weddell')])
    if CMIP6_lateral_melt_SH is not None:
        CMIP6_lateral_melt_ds = xr.merge([CMIP6_lateral_melt_SH.to_dataset(name='lateral_melt_SH')
                  ,CMIP6_lateral_melt_NH.to_dataset(name='lateral_melt_NH')
                  ,CMIP6_lateral_melt_IA.to_dataset(name='lateral_melt_IA')
                  ,CMIP6_lateral_melt_Weddell.to_dataset(name='lateral_melt_Weddell')])
    CMIP6_evap_subl_ds = xr.merge([CMIP6_evap_subl_SH.to_dataset(name='evap_subl_SH')
              ,CMIP6_evap_subl_NH.to_dataset(name='evap_subl_NH')
              ,CMIP6_evap_subl_IA.to_dataset(name='evap_subl_IA')
              ,CMIP6_evap_subl_Weddell.to_dataset(name='evap_subl_Weddell')])
    if CMIP6_dynamics_SH is not None:
        CMIP6_dynamics_ds = xr.merge([CMIP6_dynamics_SH.to_dataset(name='dynamics_SH')
                  ,CMIP6_dynamics_NH.to_dataset(name='dynamics_NH')
                  ,CMIP6_dynamics_IA.to_dataset(name='dynamics_IA')
                  ,CMIP6_dynamics_Weddell.to_dataset(name='dynamics_Weddell')])if process_CMIP6:
    CMIP6_snowfall_ds = xr.merge([CMIP6_snowfall_SH.to_dataset(name='snowfall_SH'),
                            CMIP6_snowfall_NH.to_dataset(name='snowfall_NH'),
                            CMIP6_snowfall_IA.to_dataset(name='snowfall_IA'),
                            CMIP6_snowfall_Weddell.to_dataset(name='snowfall_Weddell')])
    CMIP6_snowmelt_ds = xr.merge([CMIP6_snowmelt_SH.to_dataset(name='snowmelt_SH'),
                            CMIP6_snowmelt_NH.to_dataset(name='snowmelt_NH'),
                            CMIP6_snowmelt_IA.to_dataset(name='snowmelt_IA'),
                            CMIP6_snowmelt_Weddell.to_dataset(name='snowmelt_Weddell')])
    if CMIP6_snow_ice_snow is not None:
        CMIP6_snow_ice_snow_ds = xr.merge([CMIP6_snow_ice_snow_SH.to_dataset(name='snow_ice_snow_SH'),
                                 CMIP6_snow_ice_snow_NH.to_dataset(name='snow_ice_snow_NH'),
                                 CMIP6_snow_ice_snow_IA.to_dataset(name='snow_ice_snow_IA'),
                                 CMIP6_snow_ice_snow_Weddell.to_dataset(name='snow_ice_snow_Weddell')])
    if CMIP6_snow_dynamics is not None:
        CMIP6_snow_dynamics_ds = xr.merge([CMIP6_snow_dynamics_SH.to_dataset(name='snow_dynamics_SH'),
                                     CMIP6_snow_dynamics_NH.to_dataset(name='snow_dynamics_NH'),
                                     CMIP6_snow_dynamics_IA.to_dataset(name='snow_dynamics_IA'),
                                     CMIP6_snow_dynamics_Weddell.to_dataset(name='snow_dynamics_Weddell')])
    if CMIP6_wind_drift is not None:
        CMIP6_wind_drift_ds = xr.merge([CMIP6_wind_drift_SH.to_dataset(name='wind_drift_SH'),
                                  CMIP6_wind_drift_NH.to_dataset(name='wind_drift_NH'),
                                  CMIP6_wind_drift_IA.to_dataset(name='wind_drift_IA'),
                                  CMIP6_wind_drift_Weddell.to_dataset(name='wind_drift_Weddell')])
    if CMIP6_evap_subl_snow is not None:
        CMIP6_evap_subl_snow_ds = xr.merge([CMIP6_evap_subl_snow_SH.to_dataset(name='evap_subl_snow_SH'),
                                      CMIP6_evap_subl_snow_NH.to_dataset(name='evap_subl_snow_NH'),
                                      CMIP6_evap_subl_snow_IA.to_dataset(name='evap_subl_snow_IA'),
                                      CMIP6_evap_subl_snow_Weddell.to_dataset(name='evap_subl_snow_Weddell')])

    CMIP6_thermo_ds = xr.merge([CMIP6_thermo_SH.to_dataset(name='thermo_SH')
              ,CMIP6_thermo_NH.to_dataset(name='thermo_NH')
              ,CMIP6_thermo_IA.to_dataset(name='thermo_IA')
              ,CMIP6_thermo_Weddell.to_dataset(name='thermo_Weddell')])
    CMIP6_basal_growth_ds = xr.merge([CMIP6_basal_growth_SH.to_dataset(name='basal_growth_SH')
              ,CMIP6_basal_growth_NH.to_dataset(name='basal_growth_NH')
              ,CMIP6_basal_growth_IA.to_dataset(name='basal_growth_IA')
              ,CMIP6_basal_growth_Weddell.to_dataset(name='basal_growth_Weddell')])
    CMIP6_frazil_ds = xr.merge([CMIP6_frazil_SH.to_dataset(name='frazil_SH')
              ,CMIP6_frazil_NH.to_dataset(name='frazil_NH')
              ,CMIP6_frazil_IA.to_dataset(name='frazil_IA')
              ,CMIP6_frazil_Weddell.to_dataset(name='frazil_Weddell')])
    CMIP6_snow_ice_ds = xr.merge([CMIP6_snow_ice_SH.to_dataset(name='snow_ice_SH')
              ,CMIP6_snow_ice_NH.to_dataset(name='snow_ice_NH')
              ,CMIP6_snow_ice_IA.to_dataset(name='snow_ice_IA')
              ,CMIP6_snow_ice_Weddell.to_dataset(name='snow_ice_Weddell')])
    CMIP6_top_melt_ds = xr.merge([CMIP6_top_melt_SH.to_dataset(name='top_melt_SH')
              ,CMIP6_top_melt_NH.to_dataset(name='top_melt_NH')
              ,CMIP6_top_melt_IA.to_dataset(name='top_melt_IA')
              ,CMIP6_top_melt_Weddell.to_dataset(name='top_melt_Weddell')])
    CMIP6_basal_melt_ds = xr.merge([CMIP6_basal_melt_SH.to_dataset(name='basal_melt_SH')
              ,CMIP6_basal_melt_NH.to_dataset(name='basal_melt_NH')
              ,CMIP6_basal_melt_IA.to_dataset(name='basal_melt_IA')
              ,CMIP6_basal_melt_Weddell.to_dataset(name='basal_melt_Weddell')])
    if CMIP6_lateral_melt_SH is not None:
        CMIP6_lateral_melt_ds = xr.merge([CMIP6_lateral_melt_SH.to_dataset(name='lateral_melt_SH')
                  ,CMIP6_lateral_melt_NH.to_dataset(name='lateral_melt_NH')
                  ,CMIP6_lateral_melt_IA.to_dataset(name='lateral_melt_IA')
                  ,CMIP6_lateral_melt_Weddell.to_dataset(name='lateral_melt_Weddell')])
    CMIP6_evap_subl_ds = xr.merge([CMIP6_evap_subl_SH.to_dataset(name='evap_subl_SH')
              ,CMIP6_evap_subl_NH.to_dataset(name='evap_subl_NH')
              ,CMIP6_evap_subl_IA.to_dataset(name='evap_subl_IA')
              ,CMIP6_evap_subl_Weddell.to_dataset(name='evap_subl_Weddell')])
    if CMIP6_dynamics_SH is not None:
        CMIP6_dynamics_ds = xr.merge([CMIP6_dynamics_SH.to_dataset(name='dynamics_SH')
                  ,CMIP6_dynamics_NH.to_dataset(name='dynamics_NH')
                  ,CMIP6_dynamics_IA.to_dataset(name='dynamics_IA')
                  ,CMIP6_dynamics_Weddell.to_dataset(name='dynamics_Weddell')])

In [38]:
def _clean(ds):
    """Strip netCDF encoding (chunksizes, compression filters, etc.) inherited from
    source files before writing."""
    ds = ds.copy()
    ds.encoding = {}
    for v in ds.variables:
        ds[v].encoding = {}
    return ds

def _save_nc(ds, final_path):
    """Write to a .tmp file in the same directory as final_path, then atomically
    rename it into place with os.replace(). This is a single metadata operation on
    the same (NFS) filesystem rather than a byte-for-byte copy, so it doesn't depend
    on local /tmp disk space (writing to local disk first, then copying to NFS, hit
    'OSError: No space left on device' on this pod's small local disk). It also means
    a write that dies partway through only corrupts the .tmp file — final_path is
    never left in a truncated state, unlike writing directly over an existing file."""
    ds = _clean(ds.load())
    tmp_path = final_path + '.tmp'
    ds.to_netcdf(tmp_path)
    os.replace(tmp_path, final_path)

if save==True and process_CMIP6:
    _save_nc(CMIP6_snowfall_ds, save_path + models[0] + '_snowfall_2015_2100.nc')
    _save_nc(CMIP6_snowmelt_ds, save_path + models[0] + '_snowmelt_2015_2100.nc')
    if CMIP6_snow_dynamics is not None:
        _save_nc(CMIP6_snow_ice_snow_ds, save_path + models[0] + '_snow_ice_snow_2015_2100.nc')
    if CMIP6_snow_dynamics is not None:
        _save_nc(CMIP6_snow_dynamics_ds, save_path + models[0] + '_snow_dynamics_2015_2100.nc')
    if CMIP6_wind_drift is not None:
        _save_nc(CMIP6_wind_drift_ds, save_path + models[0] + '_wind_drift_2015_2100.nc')
    if CMIP6_evap_subl_snow is not None:
        _save_nc(CMIP6_evap_subl_snow_ds, save_path + models[0] + '_evap_subl_snow_2015_2100.nc')

    _save_nc(CMIP6_thermo_ds, save_path+models[0]+'_thermo_2015_2100.nc')
    _save_nc(CMIP6_basal_growth_ds, save_path+models[0]+'_basal_growth_2015_2100.nc')
    _save_nc(CMIP6_frazil_ds, save_path+models[0]+'_frazil_2015_2100.nc')
    _save_nc(CMIP6_snow_ice_ds, save_path+models[0]+'_snow_ice_2015_2100.nc')
    _save_nc(CMIP6_top_melt_ds, save_path+models[0]+'_top_melt_2015_2100.nc')
    _save_nc(CMIP6_basal_melt_ds, save_path+models[0]+'_basal_melt_2015_2100.nc')
    if CMIP6_lateral_melt_SH is not None:
        _save_nc(CMIP6_lateral_melt_ds, save_path+models[0]+'_lateral_melt_2015_2100.nc')
    _save_nc(CMIP6_evap_subl_ds, save_path+models[0]+'_evap_subl_2015_2100.nc')
    if CMIP6_dynamics_SH is not None:
        _save_nc(CMIP6_dynamics_ds, save_path+models[0]+'_dynamics_2015_2100.nc')
def _clean(ds):
    """Strip netCDF encoding (chunksizes, compression filters, etc.) inherited from
    source files before writing."""
    ds = ds.copy()
    ds.encoding = {}
    for v in ds.variables:
        ds[v].encoding = {}
    return ds

def _save_nc(ds, final_path):
    """Write to a .tmp file in the same directory as final_path, then atomically
    rename it into place with os.replace(). This is a single metadata operation on
    the same (NFS) filesystem rather than a byte-for-byte copy, so it doesn't depend
    on local /tmp disk space (writing to local disk first, then copying to NFS, hit
    'OSError: No space left on device' on this pod's small local disk). It also means
    a write that dies partway through only corrupts the .tmp file — final_path is
    never left in a truncated state, unlike writing directly over an existing file."""
    ds = _clean(ds.load())
    tmp_path = final_path + '.tmp'
    ds.to_netcdf(tmp_path)
    os.replace(tmp_path, final_path)

if save==True and process_CMIP6:
    _save_nc(CMIP6_snowfall_ds, save_path + models[0] + '_snowfall_2015_2100.nc')
    _save_nc(CMIP6_snowmelt_ds, save_path + models[0] + '_snowmelt_2015_2100.nc')
    if CMIP6_snow_dynamics is not None:
        _save_nc(CMIP6_snow_ice_snow_ds, save_path + models[0] + '_snow_ice_snow_2015_2100.nc')
    if CMIP6_snow_dynamics is not None:
        _save_nc(CMIP6_snow_dynamics_ds, save_path + models[0] + '_snow_dynamics_2015_2100.nc')
    if CMIP6_wind_drift is not None:
        _save_nc(CMIP6_wind_drift_ds, save_path + models[0] + '_wind_drift_2015_2100.nc')
    if CMIP6_evap_subl_snow is not None:
        _save_nc(CMIP6_evap_subl_snow_ds, save_path + models[0] + '_evap_subl_snow_2015_2100.nc')

    _save_nc(CMIP6_thermo_ds, save_path+models[0]+'_thermo_2015_2100.nc')
    _save_nc(CMIP6_basal_growth_ds, save_path+models[0]+'_basal_growth_2015_2100.nc')
    _save_nc(CMIP6_frazil_ds, save_path+models[0]+'_frazil_2015_2100.nc')
    _save_nc(CMIP6_snow_ice_ds, save_path+models[0]+'_snow_ice_2015_2100.nc')
    _save_nc(CMIP6_top_melt_ds, save_path+models[0]+'_top_melt_2015_2100.nc')
    _save_nc(CMIP6_basal_melt_ds, save_path+models[0]+'_basal_melt_2015_2100.nc')
    if CMIP6_lateral_melt_SH is not None:
        _save_nc(CMIP6_lateral_melt_ds, save_path+models[0]+'_lateral_melt_2015_2100.nc')
    _save_nc(CMIP6_evap_subl_ds, save_path+models[0]+'_evap_subl_2015_2100.nc')
    if CMIP6_dynamics_SH is not None:
        _save_nc(CMIP6_dynamics_ds, save_path+models[0]+'_dynamics_2015_2100.nc')


In [39]:
from load import clear_esgf_cache

# Free /tmp space used by the raw ESGF downloads now that the area-integrated
# outputs for this model have been written to save_path.
if save == True and process_CMIP6:
    clear_esgf_cache()

## CESM2-LE

Unlike CMIP6 models (loaded via ESGF/Pangeo), CESM2-LE data is accessed through the `CESM` class, which reads from local CryoCloud storage or THREDDS OPeNDAP. No catalog search is involved: set `local_path` to the directory containing the CESM2-LE time-series files, or enable `load_opendap` to stream remotely.

### Load gridded budget variables

In [40]:
load_opendap = False
load_locally = True
local_path = '/home/jovyan/shared-public/ICESat-2-sea-ice/better_output/LE2/b.e21'

In [41]:
if process_CESM2_LE and load_locally:
        LE2_thermo = CESM(variable='sidmassth',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_dynamics = CESM(variable='sidmassdyn',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_basal_growth = CESM(variable='sidmassgrowthbot',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_frazil = CESM(variable='sidmassgrowthwat',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_snow_ice = CESM(variable='sidmasssi',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_top_melt = CESM(variable='sidmassmelttop',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_basal_melt = CESM(variable='sidmassmeltbot',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_lateral_melt = CESM(variable='sidmasslat',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_evap_subl = CESM(variable='sidmassevapsubl',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()

        # sea ice concentration
        LE2_SIC = CESM(variable='aice',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()

        LE2_snowmelt = CESM(variable='sndmassmelt',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_snowfall = CESM(variable='sndmasssnf',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()
        LE2_evap_subl_snow = CESM(variable='sndmassubl',source_id='CESM2-LE',time_chunks=200,data_path=local_path).load_data()


In [42]:
if process_CESM2_LE and load_opendap:
        LE2_thermo = CESM(variable='sidmassth',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_dynamics = CESM(variable='sidmassdyn',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_basal_growth = CESM(variable='sidmassgrowthbot',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_frazil = CESM(variable='sidmassgrowthwat',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_snow_ice = CESM(variable='sidmasssi',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_top_melt = CESM(variable='sidmassmelttop',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_basal_melt = CESM(variable='sidmassmeltbot',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_lateral_melt = CESM(variable='sidmasslat',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_evap_subl = CESM(variable='sidmassevapsubl',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        # sea ice concentration
        LE2_SIC = CESM(variable='aice',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()

        LE2_snowmelt = CESM(variable='sndmassmelt',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_snowfall = CESM(variable='sndmasssnf',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()
        LE2_evap_subl_snow = CESM(variable='sndmassubl',source_id='CESM2-LE',time_chunks=50
                                 ,data_path=None,opendap_experiment='BSSP370',opendap_time_range=(2015,2034)).load_data()


### Preprocess

In [43]:
if process_CESM2_LE:
    # Convert ice→snow mass transfer to snow-side sign convention
    # conversion factor needed to convert ice mass to snow mass rate change
    LE2_snow_ice_snow = (-LE2_snow_ice.sidmasssi * (330/917)).to_dataset(name='sndmasssi')

    # snowmelt is stored as positive loss in CESM; negate for sign consistency
    LE2_snowmelt = -LE2_snowmelt


### Area-integrate snow fluxes (10³ Gt month⁻¹)

In [44]:
if process_CESM2_LE:
    seconds = LE2_SIC.time.dt.days_in_month * 86400

    LE2_snowmelt_SH = (((LE2_snowmelt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassmelt
    LE2_snowmelt_Weddell = ((region_mask(LE2_snowmelt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassmelt
    
    LE2_snowfall_SH = (((LE2_snowfall*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssnf
    LE2_snowfall_Weddell = ((region_mask(LE2_snowfall*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssnf
    
    LE2_snow_ice_snow_SH = (((LE2_snow_ice_snow*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssi
    LE2_snow_ice_snow_Weddell = ((region_mask(LE2_snow_ice_snow*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmasssi
    
    LE2_evap_subl_snow_SH = (((LE2_evap_subl_snow*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassubl
    LE2_evap_subl_snow_Weddell = ((region_mask(LE2_evap_subl_snow*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sndmassubl
    
    LE2_snowmelt_NH = (((LE2_snowmelt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassmelt
    LE2_snowfall_NH = (((LE2_snowfall*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssnf
    LE2_snow_ice_snow_NH = (((LE2_snow_ice_snow*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssi
    LE2_evap_subl_snow_NH = (((LE2_evap_subl_snow*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassubl
    
    LE2_snowmelt_IA = ((region_mask(LE2_snowmelt*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassmelt
    LE2_snowfall_IA = ((region_mask(LE2_snowfall*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssnf
    LE2_snow_ice_snow_IA = ((region_mask(LE2_snow_ice_snow*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmasssi
    LE2_evap_subl_snow_IA = ((region_mask(LE2_evap_subl_snow*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sndmassubl

### Save snow budget

In [45]:
if save==True and process_CESM2_LE:
    xr.merge([LE2_snowmelt_SH.to_dataset(name='snowmelt_SH')
              ,LE2_snowmelt_Weddell.to_dataset(name='snowmelt_Weddell'),
             LE2_snowmelt_NH.to_dataset(name='snowmelt_NH'),
             LE2_snowmelt_IA.to_dataset(name='snowmelt_IA')]).to_netcdf(save_path+'LE2'+'_snowmelt_2015_2034.nc')
    
    xr.merge([LE2_snowfall_SH.to_dataset(name='snowfall_SH')
              ,LE2_snowfall_Weddell.to_dataset(name='snowfall_Weddell'),
             LE2_snowfall_NH.to_dataset(name='snowfall_NH'),
             LE2_snowfall_IA.to_dataset(name='snowfall_IA')]).to_netcdf(save_path+'LE2'+'_snowfall_2015_2034.nc')
    
    xr.merge([LE2_snow_ice_snow_SH.to_dataset(name='snow_ice_snow_SH')
              ,LE2_snow_ice_snow_Weddell.to_dataset(name='snow_ice_snow_Weddell'),
             LE2_snow_ice_snow_NH.to_dataset(name='snow_ice_snow_NH'),
             LE2_snow_ice_snow_IA.to_dataset(name='snow_ice_snow_IA')]).to_netcdf(save_path+'LE2'+'_snow_ice_snow_2015_2034.nc')
    xr.merge([LE2_evap_subl_snow_SH.to_dataset(name='evap_subl_snow_SH')
              ,LE2_evap_subl_snow_Weddell.to_dataset(name='evap_subl_snow_Weddell'),
             LE2_evap_subl_snow_NH.to_dataset(name='evap_subl_snow_NH'),
             LE2_evap_subl_snow_IA.to_dataset(name='evap_subl_snow_IA')]).to_netcdf(save_path+'LE2'+'_evap_subl_snow_2015_2034.nc')

### Area-integrate ice fluxes (10³ Gt month⁻¹)

In [46]:
if process_CESM2_LE:
    LE2_thermo_SH = (((LE2_thermo*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassth
    LE2_thermo_Weddell = ((region_mask(LE2_thermo*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassth

    LE2_dynamics_SH = (((LE2_dynamics*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassdyn
    LE2_dynamics_Weddell = ((region_mask(LE2_dynamics*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassdyn

    LE2_basal_growth_SH = (((LE2_basal_growth*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot
    LE2_basal_growth_Weddell = ((region_mask(LE2_basal_growth*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot

    LE2_frazil_SH = (((LE2_frazil*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat
    LE2_frazil_Weddell = ((region_mask(LE2_frazil*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat

    LE2_snow_ice_SH = (((LE2_snow_ice*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasssi
    LE2_snow_ice_Weddell = ((region_mask(LE2_snow_ice*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasssi

    LE2_top_melt_SH = -(((LE2_top_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmelttop
    LE2_top_melt_Weddell = -((region_mask(LE2_top_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmelttop

    LE2_basal_melt_SH = -(((LE2_basal_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmeltbot
    LE2_basal_melt_Weddell = -((region_mask(LE2_basal_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassmeltbot

    LE2_lateral_melt_SH = -(((LE2_lateral_melt*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasslat
    LE2_lateral_melt_Weddell = -((region_mask(LE2_lateral_melt*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmasslat

    LE2_evap_subl_SH = (((LE2_evap_subl*LE2_SIC.areacello).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassevapsubl
    LE2_evap_subl_Weddell = ((region_mask(LE2_evap_subl*LE2_SIC.areacello,{'Antarctic':[0]}).where(LE2_SIC.lat<0).sum(['x','y']))*seconds/1e15).sidmassevapsubl



    LE2_thermo_NH = (((LE2_thermo*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassth
    LE2_dynamics_NH = (((LE2_dynamics*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassdyn
    LE2_basal_growth_NH = (((LE2_basal_growth*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot
    LE2_frazil_NH = (((LE2_frazil*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat
    LE2_snow_ice_NH = (((LE2_snow_ice*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasssi
    LE2_top_melt_NH = -(((LE2_top_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmelttop
    LE2_basal_melt_NH = -(((LE2_basal_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmeltbot
    LE2_lateral_melt_NH = -(((LE2_lateral_melt*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasslat
    LE2_evap_subl_NH = (((LE2_evap_subl*LE2_SIC.areacello).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassevapsubl

    LE2_thermo_IA = ((region_mask(LE2_thermo*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassth
    LE2_dynamics_IA = ((region_mask(LE2_dynamics*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassdyn
    LE2_basal_growth_IA = ((region_mask(LE2_basal_growth*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthbot
    LE2_frazil_IA = ((region_mask(LE2_frazil*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassgrowthwat
    LE2_snow_ice_IA = ((region_mask(LE2_snow_ice*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasssi
    LE2_top_melt_IA = -((region_mask(LE2_top_melt*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmelttop
    LE2_basal_melt_IA = -((region_mask(LE2_basal_melt*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassmeltbot
    LE2_lateral_melt_IA = -((region_mask(LE2_lateral_melt*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmasslat
    LE2_evap_subl_IA = ((region_mask(LE2_evap_subl*LE2_SIC.areacello,{'Arctic': [0,1,2,3,4,5,6,7]}).where(LE2_SIC.lat>0).sum(['x','y']))*seconds/1e15).sidmassevapsubl

### Save ice budget

In [47]:
if save==True and process_CESM2_LE:
    xr.merge([LE2_thermo_SH.to_dataset(name='thermo_SH')
              ,LE2_thermo_Weddell.to_dataset(name='thermo_Weddell'),
             LE2_thermo_NH.to_dataset(name='thermo_NH'),
             LE2_thermo_IA.to_dataset(name='thermo_IA')]).to_netcdf(save_path+'LE2'+'_thermo_2015_2034.nc')
    
    xr.merge([LE2_dynamics_SH.to_dataset(name='dynamics_SH')
              ,LE2_dynamics_Weddell.to_dataset(name='dynamics_Weddell'),
             LE2_dynamics_NH.to_dataset(name='dynamics_NH'),
             LE2_dynamics_IA.to_dataset(name='dynamics_IA')]).to_netcdf(save_path+'LE2'+'_dynamics_2015_2034.nc')
    
    xr.merge([LE2_basal_growth_SH.to_dataset(name='basal_growth_SH')
              ,LE2_basal_growth_Weddell.to_dataset(name='basal_growth_Weddell'),
             LE2_basal_growth_NH.to_dataset(name='basal_growth_NH'),
             LE2_basal_growth_IA.to_dataset(name='basal_growth_IA')]).to_netcdf(save_path+'LE2'+'_basal_growth_2015_2034.nc')
    
    xr.merge([LE2_frazil_SH.to_dataset(name='frazil_SH')
              ,LE2_frazil_Weddell.to_dataset(name='frazil_Weddell'),
             LE2_frazil_NH.to_dataset(name='frazil_NH'),
             LE2_frazil_IA.to_dataset(name='frazil_IA')]).to_netcdf(save_path+'LE2'+'_frazil_2015_2034.nc')
    
    xr.merge([LE2_snow_ice_SH.to_dataset(name='snow_ice_SH')
              ,LE2_snow_ice_Weddell.to_dataset(name='snow_ice_Weddell'),
             LE2_snow_ice_NH.to_dataset(name='snow_ice_NH'),
             LE2_snow_ice_IA.to_dataset(name='snow_ice_IA')]).to_netcdf(save_path+'LE2'+'_snow_ice_2015_2034.nc')
    
    xr.merge([LE2_top_melt_SH.to_dataset(name='top_melt_SH')
              ,LE2_top_melt_Weddell.to_dataset(name='top_melt_Weddell'),
             LE2_top_melt_NH.to_dataset(name='top_melt_NH'),
             LE2_top_melt_IA.to_dataset(name='top_melt_IA')]).to_netcdf(save_path+'LE2'+'_top_melt_2015_2034.nc')
    
    xr.merge([LE2_basal_melt_SH.to_dataset(name='basal_melt_SH')
              ,LE2_basal_melt_Weddell.to_dataset(name='basal_melt_Weddell'),
             LE2_basal_melt_NH.to_dataset(name='basal_melt_NH'),
             LE2_basal_melt_IA.to_dataset(name='basal_melt_IA')]).to_netcdf(save_path+'LE2'+'_basal_melt_2015_2034.nc')
    
    xr.merge([LE2_lateral_melt_SH.to_dataset(name='lateral_melt_SH')
              ,LE2_lateral_melt_Weddell.to_dataset(name='lateral_melt_Weddell'),
             LE2_lateral_melt_NH.to_dataset(name='lateral_melt_NH'),
             LE2_lateral_melt_IA.to_dataset(name='lateral_melt_IA')]).to_netcdf(save_path+'LE2'+'_lateral_melt_2015_2034.nc')
    
    xr.merge([LE2_evap_subl_SH.to_dataset(name='evap_subl_SH')
              ,LE2_evap_subl_Weddell.to_dataset(name='evap_subl_Weddell'),
             LE2_evap_subl_NH.to_dataset(name='evap_subl_NH'),
             LE2_evap_subl_IA.to_dataset(name='evap_subl_IA')]).to_netcdf(save_path+'LE2'+'_evap_subl_2015_2034.nc')